In [2]:
using CSV, DataFrames, Dates, JSON, XLSX, StatsBase

In [3]:
function print_dict(d::Dict)
    for (k, v) in dµ
        println("$(k) => $(v)")
    end
end

print_dict (generic function with 1 method)

In [15]:
#= folder = "../instances_xlsx/A_MTN_3/"
file = "../instances_xlsx/A_MTN_3/224FL_10A_2.xlsx" =#
file = "../OAMRP-Data/40-aircraft/875FL_40A.xlsx"
df_flights = DataFrame(XLSX.readtable(file, "Data"))
sort!(df_flights, [:TAIL_NUMBER])


Row,YEAR,MONTH,DAY,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DEPARTURE_TIME,AIR_TIME,ARRIVAL_TIME
,Any,Any,Any,Any,Any,Any,Any,Any,Any
1,2024,2,1,N224AK,DEN,SEA,1095,133.0,1196
2,2024,2,1,N224AK,PDX,SFO,461,97.0,593
3,2024,2,1,N224AK,SEA,DEN,823,140.0,1040
4,2024,2,1,N224AK,SEA,OAK,1251,104.0,1384
5,2024,2,1,N224AK,SFO,SEA,644,92.0,768
6,2024,2,2,N224AK,EWR,SEA,2557,293.0,2708
7,2024,2,2,N224AK,OAK,SEA,1791,92.0,1913
8,2024,2,2,N224AK,SEA,EWR,1973,324.0,2505
9,2024,2,2,N224AK,SEA,TUS,2790,150.0,3017


In [16]:
println(minimum(df_flights.AIR_TIME))
println(maximum(df_flights.AIR_TIME))
println(round(Int, median(df_flights.AIR_TIME)))
println(round(Int, mean(df_flights.AIR_TIME)))

21.0
395.0
176
193


In [8]:

aircraft_day_df = combine(groupby(df_flights, [:TAIL_NUMBER, :DAY]),
    :AIR_TIME => sum => :FLYING_TIME,
    :AIR_TIME => length => :TAKEOFF
)

sort!(aircraft_day_df, [:TAIL_NUMBER, :DAY])
aircraft_day_df

Row,TAIL_NUMBER,DAY,FLYING_TIME,TAKEOFF
,Any,Any,Float64,Int64
1,N253AK,1,118.0,1
2,N253AK,2,836.0,3
3,N253AK,3,884.0,3
4,N253AK,4,664.0,3
5,N253AK,5,966.0,4
6,N253AK,6,688.0,5
7,N253AK,7,629.0,5
8,N285AK,1,772.0,3
9,N285AK,2,759.0,3


In [6]:
println(minimum(aircraft_day_df.FLYING_TIME))
println(maximum(aircraft_day_df.FLYING_TIME))
println(median(aircraft_day_df.FLYING_TIME))
println(mean(aircraft_day_df.FLYING_TIME))

229
1079
676.0
650.8857142857142


In [9]:
aircraft_agg_df = combine(groupby(aircraft_day_df, :TAIL_NUMBER),
    :FLYING_TIME => sum => :FLYING_TIME_SUM,
    :FLYING_TIME => minimum => :FLYING_TIME_MIN,
    :FLYING_TIME => maximum => :FLYING_TIME_MAX,
    :FLYING_TIME => (x -> round(Int, mean(x))) => :FLYING_TIME_MEAN,
    :FLYING_TIME => (x -> round(Int, median(x))) => :FLYING_TIME_MEDIAN,
    :TAKEOFF => sum => :TAKEOFF_SUM,
    :TAKEOFF => minimum => :TAKEOFF_MIN,
    :TAKEOFF => maximum => :TAKEOFF_MAX,
    :TAKEOFF => (x -> round(Int, mean(x))) => :TAKEOFF_MEAN,
    :TAKEOFF => (x -> round(Int, median(x))) => :TAKEOFF_MEDIAN
)

Row,TAIL_NUMBER,FLYING_TIME_SUM,FLYING_TIME_MIN,FLYING_TIME_MAX,FLYING_TIME_MEAN,FLYING_TIME_MEDIAN,TAKEOFF_SUM,TAKEOFF_MIN,TAKEOFF_MAX,TAKEOFF_MEAN,TAKEOFF_MEDIAN
,Any,Float64,Float64,Float64,Int64,Int64,Int64,Int64,Int64,Int64,Int64
1,N253AK,4785.0,118.0,966.0,684,688,24,1,5,3,3
2,N285AK,5740.0,663.0,1117.0,820,772,25,2,5,4,4
3,N297AK,4504.0,205.0,852.0,643,720,26,2,5,4,4
4,N307AS,4561.0,503.0,751.0,652,670,31,4,5,4,4
5,N320AS,4480.0,508.0,836.0,640,589,33,3,6,5,5
6,N431AS,4962.0,73.0,962.0,709,808,21,1,5,3,3
7,N531AS,4824.0,386.0,951.0,689,725,26,2,5,4,4
8,N558AS,3858.0,246.0,822.0,551,549,30,3,5,4,4
9,N587AS,5019.0,648.0,837.0,717,685,29,3,6,4,4


In [10]:
println("MIN = ", minimum(aircraft_day_df.FLYING_TIME))
println("MAX = ", maximum(aircraft_day_df.FLYING_TIME))

MIN = 73.0
MAX = 1117.0


In [11]:
println("Average flying time per aircraft per day: ", round(sum(result_df.FLYING_TIME)/length(tail_numbers)/31, digits=2), " minutes")
println("Average takeoffs per aircraft per day: ", round(sum(result_df.TAKEOFF)/length(tail_numbers)/31, digits=2), " takeoffs")
println("Average flying time per takeoff: ", round(sum(result_df.FLYING_TIME)/sum(result_df.TAKEOFF), digits=2), " minutes")
#= println("Average flying time per aircraft per week: ", round(sum(result_df.FLYING_TIME)/length(tail_numbers)/4, digits=2), " minutes")
println("Average takeoffs per aircraft per week: ", round(sum(result_df.TAKEOFF)/length(tail_numbers)/4, digits=2), " minutes")
=#

UndefVarError: UndefVarError: `result_df` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [17]:
function construct_routes(xlsx_file)
    # Trier par avion et heure de départ
    df_flights = DataFrame(XLSX.readtable(xlsx_file, "Data"))
    df_sorted = sort(df_flights, [:TAIL_NUMBER, :DEPARTURE_TIME])

    # Ajouter l'encodage des vols
    transform!(df_sorted, [:ORIGIN_AIRPORT, :DESTINATION_AIRPORT, :DEPARTURE_TIME, :ARRIVAL_TIME] => 
            ((o, d, dep, arr) -> string.(o, "_", d, "_", dep, "_", arr)) => :FLIGHT_CODE)

    # Calculer le cumul des heures de vol par avion
    transform!(groupby(df_sorted, :TAIL_NUMBER), 
        :AIR_TIME => cumsum => :CUMULATIVE_FLYING_TIME)

    # Construire les itinéraires avec l'évolution du cumul
    itineraries = combine(groupby(df_sorted, :TAIL_NUMBER)) do group
        # Créer les codes de vol avec le cumul
        flight_evolution = ["$(row.FLIGHT_CODE) ($(row.CUMULATIVE_FLYING_TIME))" 
                            for row in eachrow(group)]
        
        DataFrame(
            NBR_FLIGHTS = nrow(group),
            ITINERARY = join(flight_evolution, " → "),
            TOTAL_FLYING_TIME = maximum(group.CUMULATIVE_FLYING_TIME)
        )
    end
    return itineraries
end 


construct_routes (generic function with 1 method)

In [18]:
itineraries = construct_routes(file)

open("itineraries.txt", "w") do io
    println(io, "AIRCRAFT | NBR_FLIGHTS | ROUTE | TOTAL_FLYING_TIME ")
    for r in eachrow(itineraries)
        println(io,
            r.TAIL_NUMBER, " | ",
            r.NBR_FLIGHTS, " | ",
            r.ITINERARY, " | ",
            r.TOTAL_FLYING_TIME, "\n"
        )
    end
end